# Calibration: CS-01 TAS apparatus

**Purpose**: characterise the host noise floor and the apparatus's rate ceiling under each deployment mode (`localhost`, `multiprocess`). Decides whether the host is fit to run an experimental method run; the verdict block in each envelope drives the pre-experiment gate.

**Inputs**:

- `data/config/method/experimental.json` (orchestrator + per-spawner runtime tuning).
- `data/config/method/prototype/calibration.json` (probe sample sizes + gate threshold; runtime fallbacks on the source side).

**Outputs**:

- Envelopes: `data/results/calibration/<dpl>/<host>_<run_id>.json` (one per dpl mode).
- Per-dpl figures: `data/img/calibration/<dpl>/{timer,jitter,loopback,handler_scaling,rate_sweep,summary}.{png,svg}`.
- Headline overlay: `data/img/calibration/comparison/overlay.{png,svg}`.

**Equivalent CLI**:

```bash
python -m src.methods.experimental --stage calibration --dpl localhost
python -m src.methods.experimental --stage calibration --dpl multiprocess
```


In [ ]:
%matplotlib inline

from pathlib import Path

from src.experimental.procedure.deployment import Dpl
from src.methods.experimental import run_calibration
from src.view import (plot_calibration_summary,
                      plot_envelope_overlay,
                      plot_handler_scaling,
                      plot_jitter,
                      plot_loopback,
                      plot_rate_sweep,
                      plot_timer,
                      plot_workers_scaling)

# deployment-mode axis driven side-by-side
DPLS: list[Dpl] = ["localhost", "multiprocess"]

# figure output root for this method
IMG_ROOT = Path("data/img/calibration")

# Human-readable deployment-mode labels for plot titles / legends.
DISPLAY: dict[Dpl, str] = {
    "localhost": "Localhost",
    "multiprocess": "Multiprocess",
    "remote": "Remote",
}


## 1. Run calibration on every deployment mode

`run_calibration(dpl=...)` builds the envelope skeleton, runs the four host-floor probes (timer / jitter / loopback / handler scaling), brings up the vernier under the chosen deployment mode, drives the rate sweep at workers=1, runs the workers ramp on multiprocess, applies the gate verdict, and writes the envelope to `data/results/calibration/<dpl>/`.

**Why `n_clients = 8` in the JSON?** The workers ramp drives N worker processes from one Python driver. A single-process `httpx.AsyncClient` saturates around 1-1.5k req/s on Windows; past that, the *client* becomes the bottleneck and the probe mis-attributes "client busy" as "workers saturated". Eight client processes (one per ramp wave) push the client ceiling well above any worker count we'd realistically calibrate, so the recorded efficiency curve reflects worker-side scaling and not load-generator overhead. The client is deliberately overdimensioned: we want the host-vernier interaction to fail before the client does.

Takes ~1-3 minutes per deployment mode (the rate sweep + workers ramp dominate).


In [ ]:
ENVELOPES = {}
for _dpl in DPLS:
    print(f"\n=== Running calibration: dpl={_dpl} ===")
    ENVELOPES[_dpl] = run_calibration(dpl=_dpl,
                                      framework="fastapi",
                                      write=True)


### 1a. Verdict summary

Per-dpl pass / fail with per-probe diagnostics. The notebook is the human-facing check; the orchestrator's CLI prints the same data.


In [ ]:
from src.methods.experimental import _print_calibration_report

for _dpl, _env in ENVELOPES.items():
    print(f"\n========== {DISPLAY[_dpl]} ==========")
    _print_calibration_report(_env)


## 2. Per-deployment summary figure

2x3 grid: timer, jitter, loopback, handler scaling, rate sweep, gate verdict. One figure per deployment mode.


In [ ]:
for _dpl, _env in ENVELOPES.items():
    _out_dir = IMG_ROOT / _dpl
    _title = f"Calibration: {DISPLAY[_dpl]} ({_env['host']})"
    plot_calibration_summary(_env,
                             title=_title,
                             file_path=str(_out_dir),
                             fname="summary")


## 3. Cross-deployment overlay (headline figure)

Same 2x3 grid with both envelopes overlaid in distinct colours. The dissertation's apparatus-comparison figure: same machine, different deployment shapes, different noise + saturation profiles.


In [ ]:
_labelled = {DISPLAY[_dpl]: _env for _dpl, _env in ENVELOPES.items()}
plot_envelope_overlay(_labelled,
                      title="Calibration overlay: Localhost vs Multiprocess",
                      file_path=str(IMG_ROOT / "comparison"),
                      fname="overlay")


## 4. Individual probe panels

Standalone per-probe figures saved alongside the summary so the notebook reader can inspect each probe in isolation. `handler_scaling` only renders for `localhost` runs; `workers_scaling` only for `multiprocess` runs.


In [ ]:
for _dpl, _env in ENVELOPES.items():
    _out_dir = IMG_ROOT / _dpl
    _scn = DISPLAY[_dpl]
    plot_timer(_env,
               title=f"Calibration: Timer, {_scn}",
               file_path=str(_out_dir),
               fname="timer")
    plot_jitter(_env,
                title=f"Calibration: Jitter, {_scn}",
                file_path=str(_out_dir),
                fname="jitter")
    plot_loopback(_env,
                  title=f"Calibration: Loopback, {_scn}",
                  file_path=str(_out_dir),
                  fname="loopback")
    if _env.get("handler_scaling"):
        plot_handler_scaling(_env,
                             title=f"Calibration: Handler Scaling, {_scn}",
                             file_path=str(_out_dir),
                             fname="handler_scaling")
    plot_rate_sweep(_env,
                    title=f"Calibration: Rate Sweep, {_scn}",
                    file_path=str(_out_dir),
                    fname="rate_sweep")
    if _env.get("workers_scaling"):
        plot_workers_scaling(_env,
                             title=f"Calibration: Workers Scaling, {_scn}",
                             file_path=str(_out_dir),
                             fname="workers_scaling")
